# How do you score a match between two lists of numbers?


Your music app just played you a song you love, by a band you never heard of. Somewhere, two lists of numbers got compared, and the verdict was: same taste.

A song is not one number. It's loud, fast, sad, bright, many qualities at once. Your taste is that same tangle. So the real question: how do you boil a match between two many-featured things down to a single number?

We're going to build one answer with code, cell by cell, and check it against itself as we go.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def dot(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return np.sum(a * b)


## Turn the probe and feel it

Think of a taste as an arrow. Each direction is a quality, and the length along that direction is how hard the song leans that way. Here's one amber probe (that's you) and four candidate songs, all as 2D arrows.

The rule: multiply the two arrows entry by entry, then add. That sum is the score.

Before you run this: the probe points straight along the x-axis. Which candidate do you think will score highest, and can any score go negative?

In [ ]:
probe = np.array([1.0, 0.0])

c1 = np.array([3.0, 0.0])
c2 = np.array([0.0, 3.0])
c3 = np.array([-3.0, 0.0])
c4 = np.array([2.0, 2.0])

candidates = {"c1": c1, "c2": c2, "c3": c3, "c4": c4}

scores = {name: dot(probe, vec) for name, vec in candidates.items()}
for name, score in scores.items():
    print(name, "->", score)


c1 tops out because the probe points its way, not because c1 is the longest arrow (c1, c2, and c3 are all the same length). And c3 already went negative.

Now turn the probe. Nothing about the candidates changes. Before you run this: predict which candidate takes over the top score once the probe points straight up instead.

In [ ]:
probe_turned = np.array([0.0, 1.0])

scores_turned = {name: dot(probe_turned, vec) for name, vec in candidates.items()}
for name, score in scores_turned.items():
    print(name, "->", score)

assert scores_turned["c2"] > scores_turned["c1"]
assert scores["c1"] > scores["c2"]


The candidates never moved. Only the probe's direction changed, and the ranking flipped completely. So the score is not reading what the songs *are*. It's reading how well two directions agree.

Now the trap. Before you run this: stretch c1 to three times its length, but keep it pointing the exact same way. Does its score against the original probe change?

In [ ]:
def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

c1_stretched = c1 * 3.0

score_before = dot(probe, c1)
score_after = dot(probe, c1_stretched)
print("score before stretch:", score_before)
print("score after stretch: ", score_after)

same_direction = np.isclose(cosine(probe, c1), cosine(probe, c1_stretched))
print("direction unchanged:", same_direction)

assert same_direction
assert score_after > score_before


c1 never turned. Its direction, measured by cosine, is identical before and after. But its score climbed anyway, purely because it got longer. Length cheats the meter. A big score does not mean "same song," it means "points the same way," and a long arrow can fake a high number.

## Name the machine you just used

Three probe positions, three readings. Point along a candidate, point across it, point against it. Before you run this: predict the sign of each score.

In [ ]:
along = c1      # same direction as the probe
across = c2     # perpendicular to the probe
against = c3    # opposite direction from the probe

score_along = dot(probe, along)
score_across = dot(probe, across)
score_against = dot(probe, against)

print("along:  ", score_along)
print("across: ", score_across)
print("against:", score_against)

assert score_along > 0
assert score_across == 0
assert score_against < 0


Agree, ignore, oppose. One multiply-and-add, and it behaves like a meter with a needle that swings across those three readings.

**The dot product is a similarity meter.**

Two lists of numbers go in, one honest reading of alignment comes out. If that claim were false, the assert below would fail: aligned arrows would not score highest, perpendicular arrows would not land on exactly zero, and opposed arrows would not go negative.

In [ ]:
readings = [score_along, score_across, score_against]
print("meter readings, agree -> ignore -> oppose:", readings)

assert readings[0] > readings[1] > readings[2]
assert readings[1] == 0.0


## Where the meter travels, and where it breaks

Same meter, different aisle. Swap songs for a job posting. Score yourself against what a role wants: Python hours, SQL reps, nights free. Multiply matched entries, add them up. The dot product never asked what the numbers meant.

In [ ]:
# features: [python_hours, sql_reps, nights_free]
you = np.array([8.0, 5.0, 2.0])
role = np.array([6.0, 4.0, 1.0])

fit_score = dot(you, role)
print("job fit score:", fit_score)

assert fit_score == dot(role, you)


Same function, new domain, no code changed. That's exactly why the meter travels.

Now the honest look. Before you run this: if you double the role's requirements exactly, same direction, twice the numbers, does the fit score double too?

In [ ]:
role_doubled = role * 2.0

fit_score_doubled = dot(you, role_doubled)
print("original fit score:", fit_score)
print("doubled-role score:", fit_score_doubled)

assert np.isclose(fit_score_doubled, 2.0 * fit_score)


It doubles exactly. The meter is linear in each arrow you feed it. That clean scaling is the crack: a long, loud song can win on sheer size, and a padded resume can win on sheer length, because size passes straight through the meter untouched.

## For the walk home

What is the cheapest fix that keeps the direction and forgets the size?

One thread to pull: try normalizing a vector by its own length (dividing by `np.linalg.norm`) before you take the dot product, and see what happens to the score when you stretch c1 again. Does the meter still get fooled?